In [1]:
# 导入所需库
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示error
import csv

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

## 读取与预处理数据

In [2]:
# 读取训练数据和类别映射文件
train_data = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2.csv')
class_mapping = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv')

# 去除Index列
if 'Index' in train_data.columns:
    train_data = train_data.drop(columns=['Index'])

# 合并class_mapping，将Alliance映射为num和Formation
train_data = train_data.merge(class_mapping[['Alliance', 'num', 'Formation']], on='Alliance', how='left')

# 特征、标签准备
feature_cols = [col for col in train_data.columns if col not in ['Alliance', 'num', 'Formation']]
X = train_data[feature_cols]
num_labels = train_data['num']
formation_labels = train_data['Formation']

# 编码大类标签
formation_encoder = LabelEncoder()
formation_y = formation_encoder.fit_transform(formation_labels)

# 由于使用全部数据训练，直接赋值
X_train = X
num_train = num_labels
formation_train = formation_labels
formation_y_train = formation_y

## 构建神经网络与损失函数

In [3]:
def build_model(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(x_train, y_train, num_classes, verbose=0):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    model = build_model(x_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    history = model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    n_epochs = len(history.history['loss'])
    final_loss = history.history['loss'][-1]
    final_acc = history.history['acc'][-1]
    print(f"训练轮数: {n_epochs}, 最终训练精度: {final_acc:.4f}, 最终训练损失: {final_loss:.4f}")
    return model

## 训练分类模型

In [4]:
# 获取当前日期字符串
save_date = datetime.datetime.now().strftime('%Y%m%d')
model_dir = Path(f'F:/TensorFlow/xinjiang/models{save_date}')
model_dir.mkdir(parents=True, exist_ok=True)

# ==== 工具函数 ====
def get_eng_formation_map(class_mapping_path, class_mapping_df):
    if 'Eng_Formation' not in class_mapping_df.columns:
        class_mapping_df = pd.read_csv(class_mapping_path)
    return dict(zip(class_mapping_df['Formation'], class_mapping_df['Eng_Formation']))

def save_model(model, save_dir, name):
    name = name.replace(' ', '')
    path = Path(save_dir) / f'{name}.h5'
    model.save(str(path))
    print(f"模型已保存到: {path}")
    return path

def train_and_save_all_models(X_train, num_train, formation_y_train, formation_encoder, eng_formation_map, save_dir):
    # 训练大类模型
    num_formation_classes = len(np.unique(formation_y_train))
    print(f"大类（Formation）类别数: {num_formation_classes}")
    formation_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes)
    print('大类训练结束')
    save_model(formation_model, save_dir, 'formation_model')

    # 训练并保存小类模型
    small_class_models = {}
    for idx, formation in enumerate(formation_encoder.classes_):
        mask = (formation_y_train == idx)
        X_sub = X_train[mask]
        y_sub = num_train[mask]
        num_encoder = LabelEncoder()
        y_sub_encoded = num_encoder.fit_transform(y_sub)
        n_classes = len(np.unique(y_sub_encoded))
        if n_classes == 1:
            print(f"大类[{formation}] 只有一个小类，无需训练模型")
            continue
        print(f"大类[{formation}]，小类数: {n_classes}")
        model = train_and_evaluate(X_sub, y_sub_encoded, n_classes)
        eng_name = eng_formation_map.get(formation, str(formation)).replace(' ', '')
        save_model(model, save_dir, f'small_class_model_{eng_name}')
        small_class_models[formation] = (model, num_encoder)
    print('全部模型保存完毕')
    return formation_model, small_class_models

# ==== 主流程 ====
# 获取保存目录
save_date = datetime.datetime.now().strftime('%Y%m%d')
model_dir = Path(f'F:/TensorFlow/xinjiang/models{save_date}')
model_dir.mkdir(parents=True, exist_ok=True)

# 获取英文名映射
gfm_path = r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
eng_formation_map = get_eng_formation_map(gfm_path, class_mapping)

# 训练并保存所有模型
formation_model, small_class_models = train_and_save_all_models(
    X_train, num_train, formation_y_train, formation_encoder, eng_formation_map, model_dir
)

大类（Formation）类别数: 9
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
训练轮数: 320, 最终训练精度: 0.6937, 最终训练损失: 0.0211
大类训练结束
训练轮数: 320, 最终训练精度: 0.6937, 最终训练损失: 0.0211
大类训练结束
模型已保存到: F:\TensorFlow\xinjiang\models20250629\formation_model.h5
大类[丛生草类草原]，小类数: 7
模型已保存到: F:\TensorFlow\xinjiang\models20250629\formation_model.h5
大类[丛生草类草原]，小类数: 7
训练轮数: 468, 最终训练精度: 0.9567, 最终训练损失: 0.0223
训练轮数: 468, 最终训练精度: 0.9567, 最终训练损失: 0.0223
模型已保存到: F:\TensorFlow\xinjiang\models20250629\small_class_model_TussokSteppe.h5
大类[丛生草类草甸]，小类数: 3
模型已保存到: F:\TensorFlow\xinjiang\models20250629\small_class_model_TussokSteppe.h5
大类[丛生草类草甸]，小类数: 3
训练轮数: 205, 最终训练精度: 0.9048, 最终训练损失: 0.3633
训练轮数: 205, 最终训练精度: 0.9048, 最终训练损失: 0.3633
模型已保存到: F:\TensorFlow\xinjiang\models20250629\

### 预测

In [ ]:
test_files = [f'F:/TensorFlow/xinjiang/a-y_split_1/testdata20240716_{i}.csv' for i in range(1, 6)]
output_file = f'F:/TensorFlow/xinjiang/traindata{save_date}test_pred.csv'
chunk_size = 100000  # 可根据内存调整

total_files = len(test_files)

# 先清空输出文件并写入表头
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)

for file_idx, test_file in enumerate(test_files, 1):
    print(f"正在处理文件 {file_idx}/{total_files}: {test_file}")
    chunk_count = 0
    for chunk in pd.read_csv(test_file, chunksize=chunk_size):
        chunk_count += 1
        print(f"  文件{file_idx}，第 {chunk_count} 个 chunk")

        # 先预测大类
        formation_pred_prob = formation_model.predict(chunk)
        formation_pred = np.argmax(formation_pred_prob, axis=1)

        # 小类预测
        num_pred_chunk = []
        for i in range(len(chunk)):
            x_row = chunk.iloc[[i]]
            formation_pred_idx = formation_pred[i]
            formation_name = formation_encoder.classes_[formation_pred_idx]
            mask = (formation_y_train == formation_pred_idx)
            y_sub = num_train[mask]
            if len(np.unique(y_sub)) == 1:
                num_pred_chunk.append(np.unique(y_sub)[0])
                continue
            model, num_encoder = small_class_models[formation_name]
            num_pred_prob = model.predict(x_row)
            num_pred_idx = np.argmax(num_pred_prob, axis=1)[0]
            num_pred_value = num_encoder.inverse_transform([num_pred_idx])[0]
            num_pred_chunk.append(num_pred_value)

        # 写入结果
        with open(output_file, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            for pred in num_pred_chunk:
                writer.writerow([pred])

print(f"预测完成，结果已保存到: {output_file}")

正在处理文件 1/5: F:/TensorFlow/xinjiang/a-y_split_1/testdata20240716_1.csv
  文件1，第 1 个 chunk
  文件1，第 1 个 chunk
  文件1，第 2 个 chunk
  文件1，第 2 个 chunk
  文件1，第 3 个 chunk
  文件1，第 3 个 chunk
  文件1，第 4 个 chunk
  文件1，第 4 个 chunk
  文件1，第 5 个 chunk
  文件1，第 5 个 chunk
  文件1，第 6 个 chunk
  文件1，第 6 个 chunk
  文件1，第 7 个 chunk
  文件1，第 7 个 chunk
  文件1，第 8 个 chunk
  文件1，第 8 个 chunk
  文件1，第 9 个 chunk
  文件1，第 9 个 chunk
  文件1，第 10 个 chunk
  文件1，第 10 个 chunk
  文件1，第 11 个 chunk
  文件1，第 11 个 chunk
  文件1，第 12 个 chunk
  文件1，第 12 个 chunk
  文件1，第 13 个 chunk
  文件1，第 13 个 chunk
  文件1，第 14 个 chunk
  文件1，第 14 个 chunk
  文件1，第 15 个 chunk
  文件1，第 15 个 chunk
  文件1，第 16 个 chunk
  文件1，第 16 个 chunk
  文件1，第 17 个 chunk
  文件1，第 17 个 chunk
  文件1，第 18 个 chunk
  文件1，第 18 个 chunk
  文件1，第 19 个 chunk
  文件1，第 19 个 chunk
  文件1，第 20 个 chunk
  文件1，第 20 个 chunk
  文件1，第 21 个 chunk
  文件1，第 21 个 chunk
  文件1，第 22 个 chunk
  文件1，第 22 个 chunk
  文件1，第 23 个 chunk
  文件1，第 23 个 chunk
  文件1，第 24 个 chunk
  文件1，第 24 个 chunk
  文件1，第 25 个 chunk
  文件1，第 25 个 chun

In [7]:
# 文件路径
output_file_2 = f'F:/TensorFlow/xinjiang/traindata{save_date}test_pred_ascii.txt'

# 定义块大小和输出频率
chunk_size = 8418
log_frequency = 100  # 每隔 100 个块输出一次进度

# 初始化计数器
chunk_count = 0

# 写入文件头部内容
header = """ncols 8418
nrows 5332
xllcorner 72.8
yllcorner 33
cellsize 0.0031937782513316
nodata_value -9999
"""

# 分块读取和写入
with open(output_file, "r") as csvfile, open(output_file_2, "w") as outfile:
    # 写入文件头部内容
    outfile.write(header)

    reader = csv.reader(csvfile)
    chunk = []
    for row in reader:
        chunk.append(row[0])  # 取第一列的值
        if len(chunk) == chunk_size:
            # 写入当前块到文件
            outfile.write(" ".join(chunk) + "\n")
            chunk_count += 1
            if chunk_count % log_frequency == 0:  # 每隔 log_frequency 个块输出一次进度
                print(f"已处理 {chunk_count} 个 chunk")
            chunk = []  # 清空当前块
    # 写入剩余的部分
    if chunk:
        outfile.write(" ".join(chunk) + "\n")
        chunk_count += 1
        print(f"已处理 {chunk_count} 个 chunk（最后一块）")

print(f"数据处理完成，共处理 {chunk_count} 个 chunk")

已处理 100 个 chunk
已处理 200 个 chunk
已处理 200 个 chunk
已处理 300 个 chunk
已处理 300 个 chunk
已处理 400 个 chunk
已处理 400 个 chunk
已处理 500 个 chunk
已处理 500 个 chunk
已处理 600 个 chunk
已处理 600 个 chunk
已处理 700 个 chunk
已处理 700 个 chunk
已处理 800 个 chunk
已处理 800 个 chunk
已处理 900 个 chunk
已处理 900 个 chunk
已处理 1000 个 chunk
已处理 1000 个 chunk
已处理 1100 个 chunk
已处理 1100 个 chunk
已处理 1200 个 chunk
已处理 1200 个 chunk
已处理 1300 个 chunk
已处理 1300 个 chunk
已处理 1400 个 chunk
已处理 1400 个 chunk
已处理 1500 个 chunk
已处理 1500 个 chunk
已处理 1600 个 chunk
已处理 1600 个 chunk
已处理 1700 个 chunk
已处理 1700 个 chunk
已处理 1800 个 chunk
已处理 1800 个 chunk
已处理 1900 个 chunk
已处理 1900 个 chunk
已处理 2000 个 chunk
已处理 2000 个 chunk
已处理 2100 个 chunk
已处理 2100 个 chunk
已处理 2200 个 chunk
已处理 2200 个 chunk
已处理 2300 个 chunk
已处理 2300 个 chunk
已处理 2400 个 chunk
已处理 2400 个 chunk
已处理 2500 个 chunk
已处理 2500 个 chunk
已处理 2600 个 chunk
已处理 2600 个 chunk
已处理 2700 个 chunk
已处理 2700 个 chunk
已处理 2800 个 chunk
已处理 2800 个 chunk
已处理 2900 个 chunk
已处理 2900 个 chunk
已处理 3000 个 chunk
已处理 3000 个 chunk
已处理 3100 个 chu